In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 29. B5 Project — Daily Treasury Curve Forecasting Baseline

> 最も価値のある結果が「複雑なmodelを採用しない」である場合を、failureではなく研究結論として残す。

## 学習目標

- real-data manifestからlocked testまで一つのpipelineを実行できる
- zero、mean、AR、ridge、lassoを同じ情報集合で比較できる
- direction logisticをBrierとreliabilityで評価できる
- aggregateとmethodology regimeの結果からclaimを制限できる
- 4成果物と75点gateを自己監査できる

## 前提知識

- Week 17–20の全Exit Criteria
- B1のnumerical stability、B3のclaim audit、B4のtested artifact

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 29


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)

assert treasury.quality.accepted
assert np.all(forecast.target_dates > forecast.prediction_dates)
assert np.all(np.isfinite(forecast.features))
crosses_methodology_break = (
    (forecast.prediction_dates < qt.TREASURY_METHOD_BREAK.to_datetime64())
    & (forecast.target_dates >= qt.TREASURY_METHOD_BREAK.to_datetime64())
)
assert not np.any(crosses_methodology_break)

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("rows / forecast rows:", len(rates), len(forecast.regression_target))
print("methodology-crossing targets retained:", int(crosses_methodology_break.sum()))
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
rows / forecast rows: 2750 2728
methodology-crossing targets retained: 0
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Locked instructional Project contract

この固定snapshotは教材実装時に既に観察しているため、歴史的なpre-registrationとは呼ばない。
次の規約は、今後modelやhorizonを追加するときにtest結果へ合わせて変更しないinstructional lockである。

| Field | Locked value |
|---|---|
| Prediction origin | official day-$t$ curve publication後 |
| Target | next Treasury publication 10y CMT change |
| Unit | basis points |
| Features | current curve、lag changes、20-day volatility、day-of-week |
| Selection | chronological validation only |
| Final evaluation | last 20%、one use |
| Primary metric | RMSE; MAE and rank correlation are secondary |
| Adoption gate | zero RMSEを1%以上改善し、MAEも悪化せず、period sliceで重大劣化がない |
| Prohibited | PnL、liquidity、intraday、causal claim |

In [4]:
split = qt.chronological_split(len(forecast.regression_target), gap=1)
features = forecast.features
target = forecast.regression_target
direction = forecast.direction_target

ridge_grid = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]
lasso_grid = [0.001, 0.01, 0.05, 0.1, 0.2]
selection_rows = []
for family, grid in [("ridge", ridge_grid), ("lasso", lasso_grid)]:
    for alpha in grid:
        model = (
            qt.fit_ridge(features[split.train], target[split.train], alpha=alpha)
            if family == "ridge"
            else qt.fit_lasso(features[split.train], target[split.train], alpha=alpha)
        )
        metrics = qt.regression_metrics(
            target[split.validation],
            model.predict(features[split.validation]),
        )
        selection_rows.append(
            {"family": family, "alpha": alpha, "validation_rmse_bp": metrics.rmse}
        )
selection_table = pd.DataFrame(selection_rows)
best_alpha = {
    family: float(group.loc[group["validation_rmse_bp"].idxmin(), "alpha"])
    for family, group in selection_table.groupby("family")
}
display(selection_table)
print("selected alpha:", best_alpha)

,family,alpha,validation_rmse_bp
0,ridge,0.000,7.463768
1,ridge,0.010,7.426633
2,ridge,0.100,7.356254
3,ridge,1.000,7.252169
4,ridge,10.000,7.197222
5,ridge,100.000,7.185112
6,lasso,0.001,7.449210
7,lasso,0.010,7.386291
8,lasso,0.050,7.264642
9,lasso,0.100,7.208447


selected alpha: {'lasso': 0.2, 'ridge': 100.0}


## 2. Refit before the locked test

alpha選択後、test開始前までのrowsでmodelを一度refitする。test outcomeはこの時点まで参照しない。

In [5]:
final_training = np.arange(0, split.test.min() - forecast.horizon_publications)
lag_column = forecast.feature_names.index("10y_change_lag1_bp")

models = {
    "lag1_ar": qt.fit_ridge(
        features[final_training][:, [lag_column]],
        target[final_training],
        alpha=0.0,
    ),
    "ridge": qt.fit_ridge(
        features[final_training],
        target[final_training],
        alpha=best_alpha["ridge"],
    ),
    "lasso": qt.fit_lasso(
        features[final_training],
        target[final_training],
        alpha=best_alpha["lasso"],
    ),
}

test_predictions = {
    "zero": np.zeros(split.test.size),
    "historical_mean": np.full(split.test.size, target[final_training].mean()),
    "lag1_ar": models["lag1_ar"].predict(features[split.test][:, [lag_column]]),
    "ridge": models["ridge"].predict(features[split.test]),
    "lasso": models["lasso"].predict(features[split.test]),
}
test_rows = []
for name, prediction in test_predictions.items():
    metrics = qt.regression_metrics(target[split.test], prediction)
    test_rows.append(
        {
            "model": name,
            "rmse_bp": metrics.rmse,
            "mae_bp": metrics.mae,
            "rank_corr": metrics.rank_correlation,
        }
    )
test_table = pd.DataFrame(test_rows).sort_values("rmse_bp")
display(test_table)

,model,rmse_bp,mae_bp,rank_corr
4,lasso,5.599393,4.406485,0.087902
0,zero,5.599581,4.395604,0.000000
1,historical_mean,5.604698,4.409447,0.000000
3,ridge,5.604859,4.410060,0.007058
2,lag1_ar,5.606353,4.413228,-0.020853


In [6]:
fig = go.Figure()
fig.add_scatter(
    x=pd.to_datetime(forecast.prediction_dates[split.test]),
    y=target[split.test],
    name="actual",
    mode="lines",
    line={"color": "black", "width": 1},
)
for name in ["zero", "ridge", "lasso"]:
    fig.add_scatter(
        x=pd.to_datetime(forecast.prediction_dates[split.test]),
        y=test_predictions[name],
        name=name,
        mode="lines",
    )
fig.update_layout(
    title="Locked-test next-publication 10y CMT changes",
    xaxis_title="Prediction date",
    yaxis_title="Change (bp)",
    template="plotly_white",
)
fig.show()

## 3. Direction probability

regression modelのsignと確率modelを混同しない。logisticはdirection専用にfitし、constant prevalenceと比較する。

In [7]:
logistic = qt.fit_logistic_ridge(
    features[final_training],
    direction[final_training],
    alpha=0.1,
)
direction_probability = logistic.predict_proba(features[split.test])
constant_probability = np.full(split.test.size, direction[final_training].mean())
direction_table = pd.DataFrame(
    [
        {
            "model": name,
            **qt.classification_metrics(direction[split.test], probability).__dict__,
        }
        for name, probability in [
            ("constant prevalence", constant_probability),
            ("logistic", direction_probability),
        ]
    ]
)
display(direction_table)

reliability = qt.calibration_table(direction_probability, direction[split.test], n_bins=8)
fig = go.Figure()
fig.add_scatter(
    x=reliability["mean_probability"],
    y=reliability["observed_frequency"],
    mode="lines+markers",
    text=reliability["count"],
    name="logistic",
)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line={"dash": "dash"}, name="perfect")
fig.update_layout(
    title="Locked-test direction reliability",
    xaxis_title="Mean probability",
    yaxis_title="Observed frequency",
    template="plotly_white",
)
fig.show()

,model,log_loss,brier_score,accuracy,expected_calibration_error
0,constant prevalence,0.688660,0.247760,0.547619,0.005207
1,logistic,0.694283,0.250548,0.527473,0.014919


## 4. Regime and claim audit

final testはmethodology change後だけである。そこで、selectionとは独立したexpanding foldsを使い、pre/post-breakのerrorも記録する。

In [8]:
folds = qt.expanding_window_splits(
    len(target),
    initial_train_size=1000,
    test_size=250,
    step=250,
    gap=1,
)
regime_rows = []
for fold_number, (train_indices, test_indices) in enumerate(folds, start=1):
    model = qt.fit_ridge(features[train_indices], target[train_indices], alpha=best_alpha["ridge"])
    prediction = model.predict(features[test_indices])
    for regime in np.unique(forecast.methodology_regime[test_indices]):
        selected = forecast.methodology_regime[test_indices] == regime
        metrics = qt.regression_metrics(target[test_indices][selected], prediction[selected])
        zero_metrics = qt.regression_metrics(
            target[test_indices][selected],
            np.zeros(np.sum(selected)),
        )
        regime_rows.append(
            {
                "fold": fold_number,
                "regime": regime,
                "rows": int(np.sum(selected)),
                "ridge_rmse_bp": metrics.rmse,
                "zero_rmse_bp": zero_metrics.rmse,
            }
        )
regime_table = pd.DataFrame(regime_rows)
display(regime_table)

zero_row = test_table.loc[test_table["model"] == "zero"].iloc[0]
zero_rmse = float(zero_row["rmse_bp"])
zero_mae = float(zero_row["mae_bp"])
best_nonzero = test_table[test_table["model"] != "zero"].iloc[0]
relative_rmse_improvement = float(
    (zero_rmse - best_nonzero["rmse_bp"]) / zero_rmse
)
model_selected = bool(
    relative_rmse_improvement >= 0.01
    and best_nonzero["mae_bp"] <= zero_mae
)
claim = (
    f"candidate selected: {best_nonzero['model']}"
    if model_selected
    else "no model selected: no candidate clears the locked materiality gate"
)
print("best relative RMSE improvement:", relative_rmse_improvement)
print("primary claim:", claim)

,fold,regime,rows,ridge_rmse_bp,zero_rmse_bp
0,1,hermite-spline,250,4.326609,4.315553
1,2,hermite-spline,250,5.465083,5.464064
2,3,hermite-spline,211,4.383567,4.382754
3,3,monotone-convex,39,4.326654,4.320494
4,4,monotone-convex,250,8.019533,8.019227
5,5,monotone-convex,250,7.281824,7.283680
6,6,monotone-convex,250,5.689102,5.693154


best relative RMSE improvement: 3.3624156855200976e-05
primary claim: no model selected: no candidate clears the locked materiality gate


## 5. Block成果物と75点gate

| 成果物 | 必須内容 |
|---|---|
| Derivation note | empirical risk、ridge/lasso、logistic、Brier |
| Implementation + tests | snapshot loader、feature/target、split、regularized solver |
| Experiment | baseline、alpha selection、locked test、calibration、break audit |
| Technical memo | question、method、result、failure、no-selectionを含む結論 |

## 6. 失敗モード

- locked testを見てalpha、feature、horizonを再調整する
- zeroと統計的に同等の差を「改善」と強調する
- final testがpost-breakだけであることを隠す
- 1% gateを経済的materialityと呼ぶ（これは教材用の採用停止規則にすぎない）
- direction accuracyをprofitabilityへ読み替える
- modelが選ばれない結果を削除する

## 7. 段階別演習

### 基礎

1. manifest hashとrow countをmemoへ転記せよ。
2. test tableからzeroに対するRMSE差を計算せよ。

### 標準

3. 5公表観測先targetを別のlocked secondary experimentとして実行せよ。
4. bootstrapではなく時系列blockを保つ誤差差のuncertainty方法を設計せよ。

### 研究

5. methodology break後だけでtrainingを始める感度分析をsecondaryとして追加せよ。
6. executable instrumentとcost dataを追加する場合の新しいestimandを定義せよ。

## 8. Exit Criteria

- [ ] source、hash、grain、availabilityを保存した
- [ ] targetとfeature timestampのstrict orderingをtestした
- [ ] alpha selectionとlocked testを分離した
- [ ] zero、mean、AR、ridge、lassoを同じtestで比較した
- [ ] direction probabilityをBrierとreliabilityで評価した
- [ ] 1% RMSE・non-worse MAE gateを固定し、no model selectedを許容した
- [ ] 4成果物、75点、必須gateを別々に確認した

## 9. 出典

- [U.S. Treasury Daily Rates](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?page=1&type=daily_treasury_yield_curve)
- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)
- [ISLP](https://www.statlearning.com/) — baseline、regularization、classification、validation
- [The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/) — statistical learningの理論